# 03c — Create the MAG7 AI/BI Dashboard (OPTIONAL, pro mode)

Creates and **publishes** a Lakeview (AI/BI) dashboard over `ticker_data` for the pro
chatbot app's **Dashboard tab**. Published with embedding so it renders in the app
(`/embed/dashboardsv3/<id>`). The pro deploy notebook (`06b`) discovers it by name.

Every page opens with a You.com banner built from the official mark the app ships
(`chatbot-app/client/public/you-mark.png`). The last cell also swaps that banner into any
other dashboard of yours with `ydc_banner*` image widgets and republishes it.

Prereqs: `00_tables_setup` (creates `ticker_data`). Run before `06b`.
Embedding also requires the workspace setting **Settings → Security → Embed dashboards →
Allow approved domains → `*.databricksapps.com`** (so the iframe isn't blocked by CSP).

In [ ]:
exec(open('../config.py').read())
import json
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
me = w.current_user.me().user_name
# Pick a SQL warehouse (prefer serverless) unless sql_warehouse_id is pinned in config.py
wh = (sql_warehouse_id or "").strip()
if not wh:
    whs = list(w.warehouses.list())
    pick = next((x for x in whs if getattr(x,'enable_serverless_compute',False)), (whs or [None])[0])
    wh = getattr(pick,'id','')
assert wh, "No SQL warehouse found; set sql_warehouse_id in config.py"
T = f"{catalog}.{schema}.ticker_data"
print("warehouse:", wh, "| table:", T)

In [ ]:
# You.com banner for the top of every dashboard page.
# The mark is the official asset the app ships (chatbot-app/client/public/you-mark.png),
# turned white by an SVG filter so it reads on the purple, as on You.com's own app tile.
# 1540x130 matches a full-width, 3-row widget, so SIZE_MODE_FIT fills it edge to edge.
import base64, html

BANNER_TITLE = "Financial Research Agent"
BANNER_SUBTITLE = "Filings, market data and live web research · Powered by You.com"

with open('../chatbot-app/client/public/you-mark.png', 'rb') as f:
    _mark = "data:image/png;base64," + base64.b64encode(f.read()).decode()

_font = "Inter,Helvetica,Arial,sans-serif"
_svg = f"""<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" viewBox="0 0 1540 130">
<defs>
<linearGradient id="bg" x1="0%" y1="0%" x2="100%" y2="100%"><stop offset="0%" stop-color="#6B4EE6"/><stop offset="100%" stop-color="#3A1D8A"/></linearGradient>
<filter id="white"><feColorMatrix type="matrix" values="0 0 0 0 1  0 0 0 0 1  0 0 0 0 1  0 0 0 1 0"/></filter>
</defs>
<rect width="1540" height="130" rx="16" fill="url(#bg)"/>
<image x="36" y="21" width="88" height="88" href="{_mark}" xlink:href="{_mark}" filter="url(#white)"/>
<text x="150" y="66" font-family="{_font}" font-size="40" font-weight="600" fill="#FFFFFF">{html.escape(BANNER_TITLE)}</text>
<text x="150" y="99" font-family="{_font}" font-size="19" fill="#FFFFFF" fill-opacity="0.9">{html.escape(BANNER_SUBTITLE)}</text>
</svg>"""
BANNER_URL = "data:image/svg+xml;base64," + base64.b64encode(_svg.encode("utf-8")).decode()
BANNER_ALT = f"{BANNER_TITLE} - powered by You.com"


def banner_widget(name, width):
    """Image widget spanning the top `width` columns (6 on the legacy grid, 12 on GRID_V1)."""
    return {"widget": {"name": name, "imageSpec": {
                "image": {"source": {"url": BANNER_URL}, "altText": BANNER_ALT},
                "frame": {},
                "style": {"sizeMode": "SIZE_MODE_FIT", "horizontalAlign": "HORIZONTAL_ALIGN_LEFT",
                          "verticalAlign": "VERTICAL_ALIGN_CENTER"}}},
            "position": {"x": 0, "y": 0, "width": width, "height": 3}}

print(f"Banner: {BANNER_TITLE!r}, {len(BANNER_URL) // 1024} KB data URL")

In [ ]:
DASH_NAME = "MAG7 Market Dashboard"
ds_latest = ("SELECT company_name, CAST(market_cap AS DOUBLE) AS market_cap, "
  "CAST(beta AS DOUBLE) AS beta, CAST(pe_trailing AS DOUBLE) AS pe_trailing, "
  "CAST(pe_forward AS DOUBLE) AS pe_forward, CAST(ev_ebitda AS DOUBLE) AS ev_ebitda "
  f"FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY company_name ORDER BY Date DESC) AS rn FROM {T}) WHERE rn = 1")
ds_ts = (f"SELECT Date AS trade_date, company_name, CAST(price_close AS DOUBLE) AS price_close, "
  f"CAST(volume AS DOUBLE) AS volume FROM {T}")

bar={"widget":{"name":"mktcap_bar","queries":[{"name":"main_query","query":{"datasetName":"ds_latest","fields":[{"name":"company_name","expression":"`company_name`"},{"name":"sum(market_cap)","expression":"SUM(`market_cap`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"bar","encodings":{"x":{"fieldName":"company_name","scale":{"type":"categorical","sort":{"by":"y-reversed"}},"displayName":"Company"},"y":{"fieldName":"sum(market_cap)","scale":{"type":"quantitative"},"displayName":"Market cap"},"label":{"show":True}},"frame":{"showTitle":True,"title":"Latest market cap by company"},"mark":{"colors":["#FF3621","#FFAB00","#00A972","#8BCAE7","#AB4057","#99DDB4","#FCA4A1"]}}},"position":{"x":0,"y":3,"width":3,"height":6}}
line={"widget":{"name":"close_line","queries":[{"name":"main_query","query":{"datasetName":"ds_ts","fields":[{"name":"daily(trade_date)","expression":"DATE_TRUNC(\"DAY\", `trade_date`)"},{"name":"company_name","expression":"`company_name`"},{"name":"avg(price_close)","expression":"AVG(`price_close`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"line","encodings":{"x":{"fieldName":"daily(trade_date)","scale":{"type":"temporal"},"displayName":"Date"},"y":{"fieldName":"avg(price_close)","scale":{"type":"quantitative"},"displayName":"Close price"},"color":{"fieldName":"company_name","scale":{"type":"categorical"},"displayName":"Company"}},"frame":{"showTitle":True,"title":"Close price over time"}}},"position":{"x":3,"y":3,"width":3,"height":6}}
area={"widget":{"name":"vol_area","queries":[{"name":"main_query","query":{"datasetName":"ds_ts","fields":[{"name":"daily(trade_date)","expression":"DATE_TRUNC(\"DAY\", `trade_date`)"},{"name":"company_name","expression":"`company_name`"},{"name":"sum(volume)","expression":"SUM(`volume`)"}],"disaggregated":False}}],"spec":{"version":3,"widgetType":"area","encodings":{"x":{"fieldName":"daily(trade_date)","scale":{"type":"temporal"},"displayName":"Date"},"y":{"fieldName":"sum(volume)","scale":{"type":"quantitative"},"displayName":"Volume"},"color":{"fieldName":"company_name","scale":{"type":"categorical"},"displayName":"Company"}},"frame":{"showTitle":True,"title":"Trading volume over time"}}},"position":{"x":0,"y":9,"width":6,"height":5}}
table={"widget":{"name":"fundamentals_tbl","queries":[{"name":"main_query","query":{"datasetName":"ds_latest","fields":[{"name":"company_name","expression":"`company_name`"},{"name":"market_cap","expression":"`market_cap`"},{"name":"beta","expression":"`beta`"},{"name":"pe_trailing","expression":"`pe_trailing`"},{"name":"pe_forward","expression":"`pe_forward`"},{"name":"ev_ebitda","expression":"`ev_ebitda`"}],"disaggregated":True}}],"spec":{"version":2,"widgetType":"table","encodings":{"columns":[{"fieldName":"company_name","displayName":"Company"},{"fieldName":"market_cap","displayName":"Market cap"},{"fieldName":"beta","displayName":"Beta"},{"fieldName":"pe_trailing","displayName":"P/E (trailing)"},{"fieldName":"pe_forward","displayName":"P/E (forward)"},{"fieldName":"ev_ebitda","displayName":"EV/EBITDA"}]},"frame":{"showTitle":True,"title":"Latest fundamentals"}}},"position":{"x":0,"y":14,"width":6,"height":6}}
dash={"datasets":[{"name":"ds_latest","displayName":"Latest per company","queryLines":[ds_latest]},{"name":"ds_ts","displayName":"Daily ticker time series","queryLines":[ds_ts]}],"pages":[{"name":"overview","displayName":"Market Overview","pageType":"PAGE_TYPE_CANVAS","layout":[banner_widget("ydc_banner",6),bar,line,area,table]}],"uiSettings":{"applyModeEnabled":False}}

# Create (or reuse) the dashboard, then publish with embedding.
existing = next((d for d in w.api_client.do("GET","/api/2.0/lakeview/dashboards").get("dashboards",[])
                 if d.get("display_name")==DASH_NAME and d.get("lifecycle_state")!="TRASHED"), None)
if existing:
    did = existing["dashboard_id"]
    w.api_client.do("PATCH", f"/api/2.0/lakeview/dashboards/{did}", body={"serialized_dashboard":json.dumps(dash)})
    print("Updated existing dashboard:", did)
else:
    res = w.api_client.do("POST","/api/2.0/lakeview/dashboards", body={"display_name":DASH_NAME,"warehouse_id":wh,"parent_path":f"/Users/{me}","serialized_dashboard":json.dumps(dash)})
    did = res["dashboard_id"]; print("Created dashboard:", did)

w.api_client.do("POST", f"/api/2.0/lakeview/dashboards/{did}/published", body={"embed_credentials":True,"warehouse_id":wh})
print(f"✅ Published MAG7 dashboard (embed enabled). dashboard_id = {did}")
print("06b discovers it by name; or set aibi_dashboard_id in config.py.")

In [ ]:
# Refresh the banner on other dashboards that carry it (e.g. a hand-built multi-page
# dashboard): every image widget named ydc_banner* gets the new image, then the dashboard
# is republished with its own warehouse and embed setting. Only your own dashboards are
# changed - the ones under /Users/<you>/. (The list API omits paths, so each is fetched.)
# Listed over REST: the SDK preinstalled on serverless can be too old for w.lakeview.list().
dash_ids, token = [], None
while True:
    resp = w.api_client.do("GET", "/api/2.0/lakeview/dashboards",
                           query={"page_size": 100, **({"page_token": token} if token else {})})
    dash_ids += [x["dashboard_id"] for x in resp.get("dashboards", [])]
    token = resp.get("next_page_token")
    if not token:
        break

refreshed = []
for dash_id in dash_ids:
    if dash_id == globals().get("did"):
        continue
    full = w.api_client.do("GET", f"/api/2.0/lakeview/dashboards/{dash_id}")
    if not (full.get("path") or "").startswith(f"/Users/{me}/"):
        continue
    spec = json.loads(full.get("serialized_dashboard") or "{}")
    hits = 0
    for page in spec.get("pages", []):
        for item in page.get("layout", []):
            widget = item.get("widget", {})
            image = widget.get("imageSpec", {}).get("image")
            if image is not None and widget.get("name", "").startswith("ydc_banner"):
                image["source"] = {"url": BANNER_URL}
                image["altText"] = BANNER_ALT
                hits += 1
    if not hits:
        continue
    w.api_client.do("PATCH", f"/api/2.0/lakeview/dashboards/{dash_id}",
                    body={"serialized_dashboard": json.dumps(spec)})
    try:
        pub = w.api_client.do("GET", f"/api/2.0/lakeview/dashboards/{dash_id}/published")
    except Exception:
        pub = None  # draft only - nothing to republish
    if pub:
        w.api_client.do("POST", f"/api/2.0/lakeview/dashboards/{dash_id}/published",
                        body={"embed_credentials": pub.get("embed_credentials", False),
                              "warehouse_id": pub.get("warehouse_id") or full.get("warehouse_id")})
    refreshed.append(f"{full.get('display_name')} ({hits} banner(s){', republished' if pub else ''})")
print("Banner refreshed on:", refreshed or "no other dashboards")